In [190]:
import pandas as pd
import numpy as np
from EssSimulation_withoutMaxDemand import EssSimulationModel
import calendar
import copy

In [191]:
exp_name = "estimate830"
month_num = 9
node_name = "route_B_{:02d}".format(month_num)

In [192]:
es_info = {"transform_capacity": 63000,
           "invertband": 0,
           "soc_redundant_ratio": 0,
           "usable_depth": 0.97,
           "charge_loss": 0.92,
           "discharge_loss": 0.95,
           "es_charge_max": 9000,
           "es_charge_min": -9000,
           "es_capacity_max": 18000,
           "es_capacity_min": 0}

In [193]:
ratio_result_list = []
for ratio in range(10, 240, 10):
    demand_load_df = pd.read_csv(f"./data/{exp_name}/{node_name}/opt_result/demand_load.csv")
    demand_load_df['time'] = pd.to_datetime(demand_load_df['time'])
    demand_load_df.set_index('time', inplace=True)

    strategy_df = pd.read_csv(f"./data/{exp_name}/{node_name}/opt_result/ratio_experiment_dod97/schedule_result_fixline_up{ratio}.csv")
    strategy_df.rename(columns={"power_opt": "value"}, inplace=True)
    strategy_df['time'] = pd.to_datetime(strategy_df['time'])
    strategy_df.set_index('time', inplace=True)

    ele_price_df = pd.read_csv(f"./data/{exp_name}/{node_name}/opt_result/ele_price.csv")
    ele_price_df['time'] = pd.to_datetime(ele_price_df['time'])
    ele_price_df.set_index('time', inplace=True)

    simulation_model = EssSimulationModel(es_info)
    es_charge_df, es_soc_df, total_load_df = simulation_model.simulation_process(demand_load_df, strategy_df, 0)
    origin_balance, opt_balance = simulation_model.revenue_calculation(demand_load_df, es_charge_df, ele_price_df, 38.4)
    ori_max_demand = total_load_df["total_load"].mean() * 1.1
    opt_max_demand = total_load_df["total_load"].max()
    max_demand_lift_cost = (opt_max_demand - ori_max_demand) * 38.4
    gross_income = origin_balance - opt_balance - max_demand_lift_cost
    ratio_result_list.append((ratio, gross_income))
    print(f"突破比例{ratio}%, 收益为{gross_income}")
    # print(round(origin_balance - opt_balance,0))

突破比例10%, 收益为245762.50991788812
突破比例20%, 收益为264949.67574614106
突破比例30%, 收益为283786.98509379936
突破比例40%, 收益为302618.4922759857
突破比例50%, 收益为320861.7138002221
突破比例60%, 收益为337332.4922412535
突破比例70%, 收益为351205.71608738624
突破比例80%, 收益为363991.23371053365
突破比例90%, 收益为375982.0463478379
突破比例100%, 收益为385761.9142377635
突破比例110%, 收益为392812.5879707957
突破比例120%, 收益为397720.3766142019
突破比例130%, 收益为401954.30990827596
突破比例140%, 收益为404642.68440702936
突破比例150%, 收益为406730.4410564261
突破比例160%, 收益为408081.04934027157
突破比例170%, 收益为409177.79502196633
突破比例180%, 收益为410011.69089606625
突破比例190%, 收益为410231.6495952402
突破比例200%, 收益为410192.7985736024
突破比例210%, 收益为409835.95926088153
突破比例220%, 收益为409261.02294562524
突破比例230%, 收益为408550.33945607295


In [194]:
max_ratio_tuple = max(ratio_result_list, key=lambda x: x[1])

In [195]:
max_ratio = max_ratio_tuple[0]
demand_load_df = pd.read_csv(f"./data/{exp_name}/{node_name}/opt_result/demand_load.csv")
demand_load_df['time'] = pd.to_datetime(demand_load_df['time'])
demand_load_df.set_index('time', inplace=True)

strategy_df = pd.read_csv(f"./data/{exp_name}/{node_name}/opt_result/ratio_experiment_dod97/schedule_result_fixline_up{max_ratio}.csv")
strategy_df.rename(columns={"power_opt": "value"}, inplace=True)
strategy_df['time'] = pd.to_datetime(strategy_df['time'])
strategy_df.set_index('time', inplace=True)

ele_price_df = pd.read_csv(f"./data/{exp_name}/{node_name}/opt_result/ele_price.csv")
ele_price_df['time'] = pd.to_datetime(ele_price_df['time'])
ele_price_df.set_index('time', inplace=True)

simulation_model = EssSimulationModel(es_info)
es_charge_df, es_soc_df, total_load_df = simulation_model.simulation_process(demand_load_df, strategy_df, 0)
origin_balance, opt_balance = simulation_model.revenue_calculation(demand_load_df, es_charge_df, ele_price_df, 38.4)

In [196]:
ori_max_demand = total_load_df["total_load"].mean() * 1.1
opt_max_demand = total_load_df["total_load"].max()
max_demand_lift_cost = (opt_max_demand - ori_max_demand) * 38.4
gross_income = origin_balance - opt_balance - max_demand_lift_cost

print("调度后最大需量：", opt_max_demand,
      "原始最大需量：", ori_max_demand,
      "需量抬升成本：", max_demand_lift_cost,
      "总收益：", gross_income)

调度后最大需量： 12737.76 原始最大需量： 10669.250214307052 需量抬升成本： 79430.7757706092 总收益： 410231.6495952402
